Use before everthing

In [1]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


Advanced pick up

In [2]:
import time
import math

# Constants
CENTER_X = 320
GAP_TOLERANCE = 10  # 10 pixel leniency - "good enough" for the grab

def safe_get_tag():
    """Returns tag info if 1 or more tags are seen, else None."""
    try:
        data = got.get_apriltag_total_info()
        if data and len(data) > 0:
            return data
    except:
        return None
    return None

def AP_advanced_hunt(target_distance=0.16, fwd_speed=4, strafe_speed=5):
    print("Mission Started: Searching with Snake Pattern...")
    
    start_time = time.time()
    
    # 1. & 2. INITIAL MOVEMENT & DETECTION LOOP
    while True:
        AP_info = safe_get_tag()
        
        # INSTANT STOP: If we see it for even 1 frame, break the search
        if AP_info:
            got.mecanum_stop()
            print("Tag Spotted! Stopping search for adjustment.")
            time.sleep(0.3) 
            break
        
        # SNAKE SEARCH LOGIC:
        # Move forward while swaying left and right using a sine wave
        elapsed = time.time() - start_time
        # Adjust '2.0' to change sway speed, '12' to change sway width
        sway = math.sin(elapsed * 1.8) * 40
        
        got.mecanum_move_xyz(int(sway), fwd_speed, 0)
        time.sleep(0.05)

    # 3. RE-VERIFY & ADJUST (Left/Right Only - STAYS THE SAME)
    print("Adjusting alignment...")
    while True:
        AP_info = safe_get_tag()
        
        if not AP_info:
            print("Tag lost! Tiny wiggle to recover...")
            got.mecanum_move_xyz(4, 0, 0) 
            time.sleep(0.1)
            continue

        AP_x = AP_info[0][1]
        error_x = AP_x - CENTER_X

        # CHECK TOLERANCE
        if abs(error_x) <= GAP_TOLERANCE:
            print(f"Aligned (Error: {error_x}). Closing in...")
            got.mecanum_stop()
            break
        
        # Move only left or right to align
        side_move = strafe_speed if error_x > 0 else -strafe_speed
        got.mecanum_move_xyz(int(side_move), 0, 0)
        time.sleep(0.05)

    # 4. FINAL APPROACH (STAYS THE SAME)
    print("Moving to grab distance...")
    while True:
        AP_info = safe_get_tag()
        if not AP_info: break 
        
        dist = AP_info[0][6]
        if dist <= target_distance:
            got.mecanum_stop()
            break
        
        got.mecanum_move_xyz(0, 5, 0)
        time.sleep(0.05)

def advanced_pick_up():
    print("Executing Grab Sequence...")
    
    # Prepare Arm
    got.mechanical_single_joint_control(joint=2, angle=15, duration=500)
    got.mechanical_single_joint_control(joint=3, angle=-75, duration=500)
    got.mechanical_clamp_release()
    time.sleep(1.0)

    # Get final coordinates for the arm
    AP_info = safe_get_tag()
    if not AP_info:
        print("Tag lost, using last known position.")
        joint1, joint3 = 0, 10
    else:
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]
        
        joint1 = int(max(min((AP_x - CENTER_X) * -0.12, 90), -90))
        joint3 = int(max(min(AP_distance * 100 - 85, 90), -90))

    print(f"[GRAB] J1={joint1}, J3={joint3}")
    
    # Reach, nudge forward, clamp, and lift
    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1)
    # Use your specific translation speed command for the nudge
    got.mecanum_translate_speed_times(angle=0, speed=2, times=3, unit=1)
    time.sleep(1)
    got.mechanical_clamp_close()
    time.sleep(1.0)
    
    # Lift up to safe position
    got.mechanical_joint_control(0, 30, 30, 1000)
    print("Object Secured.")

# ==============================
# MAIN EXECUTION
# ==============================
try:
    # Hunt includes the new sway/snake search
    AP_advanced_hunt()
    # Grab includes your specific joint and translate commands
    advanced_pick_up()

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Manual Stop.")
except Exception as e:
    print(f"Error: {e}")
    got.mecanum_stop()

Mission Started: Searching with Snake Pattern...
Tag Spotted! Stopping search for adjustment.
Adjusting alignment...
Aligned (Error: -3.8100000000000023). Closing in...
Moving to grab distance...
Executing Grab Sequence...
[GRAB] J1=-2, J3=-63
Object Secured.


Basic Pick up

In [7]:
import time

def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drives toward an AprilTag with safety checks to prevent IndexErrors.
    """
    print("Approaching target...")
    
    while True:
        AP_info = got.get_apriltag_total_info()
        
        # ERROR FIX 1: Check if the list is empty before accessing index [0]
        if not AP_info or len(AP_info) == 0:
            print("Tag lost! Searching...")
            got.mecanum_stop()
            time.sleep(0.1) 
            continue # Skip to the next loop iteration to try and find it again

        # Safely extract data now that we know it exists
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # ERROR FIX 2: Combined movement for smoother pathing
        current_strafe = 0
        if AP_x < 320 - gap:
            current_strafe = -strafe_spd
        elif AP_x > 320 + gap:
            current_strafe = strafe_spd

        if AP_distance > distance:
            # Move forward and strafe at the same time
            got.mecanum_move_xyz(current_strafe, fwd_spd, 0)
        else:
            # We reached the goal
            got.mecanum_stop()
            print("Target reached. Stabilizing...")
            time.sleep(1.0) # ERROR FIX 3: Let inertia settle before picking up
            break

def pick_up():
    """
    Executes pickup sequence with validation logic.
    """
    AP_info = got.get_apriltag_total_info()
    
    # Validation: Ensure tag is still there before moving arm
    if not AP_info:
        print("Error: Tag disappeared right before pickup!")
        return

    AP_x = AP_info[0][1]
    AP_distance = AP_info[0][6]

    # 1. Prepare Arm
    # Using a safe mid-point for Joint 2 (30) so it doesn't hit the floor
    got.mechanical_joint_control(0, 30, 30, 1000)
    got.mechanical_clamp_release()
    time.sleep(1.5)

    # 2. Calculate Pose
    # Added 'max/min' clipping to keep values in safe mechanical ranges
    joint1 = int(max(min((AP_x - 320) * -0.1, 90), -90))
    joint3 = int(max(min(AP_distance * 100 - 80, 90), -90))

    # 3. Reach and Grab
    print(f"Executing Grab -> J1: {joint1}, J3: {joint3}")
    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1.2)
    
    got.mechanical_clamp_close()
    time.sleep(1.5)

    # 4. Retract to safe carry position
    got.mechanical_joint_control(0, 30, 30, 1000)
    print("Pickup complete.")

# Main Execution
try:
    AP_centralization_approaching()
    pick_up()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    got.mecanum_stop()


try:
    while True:
        # Poll the camera for any visible AprilTags.
        tag = got.get_apriltag_total_info()

        if tag:
            # A tag is visible — approach it and pick up the object.
            # Lower speeds (fwd_spd=5, strafe_spd=7) for more precise final approach.
            AP_centralization_approaching(distance= 1, gap=20, fwd_spd=5, strafe_spd=7)
            pick_up()

            print("Picked up!")
            break  # Exit after one successful pick-and-place

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")

Approaching target...
An unexpected error occurred: name 'got' is not defined


NameError: name 'got' is not defined

Reset

In [25]:
import time
got.mechanical_clamp_release()
time.sleep(2)
got.mechanical_joint_control(angle1=0, angle2=90, angle3=90, duration=500)
got.mechanical_clamp_close() # Close the gripper
got.mecanum_translate_speed_times(angle=180, speed=30, times=50, unit=1)



All functions so far

In [ ]:
#  Gripper control 
got.mechanical_clamp_release() # Open the gripper
got.mechanical_clamp_close() # Close the gripper

#  Arm reset 
# Returns all arm joints to their default/home position.
got.mechanical_arms_restory()

#  Move all arm joints simultaneously 
# angle1/2/3 are target angles (degrees); duration is movement time in ms.
got.mechanical_joint_control(angle1=0, angle2=0, angle3=0, duration=500)

#  Move a single arm joint 
# joint: 1=base, 2=middle, 3=furthest; angle in degrees; duration in ms.
got.mechanical_single_joint_control(joint=1, angle=0, duration=500)

In [ ]:
got.mechanical_single_joint_control(joint=2, angle=15, duration=500)
got.mechanical_single_joint_control(joint=3, angle=-75, duration=500)
